# Figure 2 — behaviour and model

Run cells in order. Calculated panels are saved under `figures/`.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "notebooks").is_dir() and (p / "data").is_dir())
sys.path.insert(0, str(ROOT / "src"))
FIGURES = ROOT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
SAVED_RESULTS = ROOT / "saved_results"

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
plt.style.use(ROOT / "notebooks/ncb.mplstyle")
plt.close("all")
np.random.seed(0)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from metpy.interpolate import interpolate_to_points

from Gbim_import_dat import get_Gbim_data_chirpfield, get_Gbim_data_DC
import plotting as p

## Load behavioral responses

In [ ]:
[CDUR, CPAU, CPER, DCs, NPUL, rXY]= get_Gbim_data_chirpfield()
[CDUR_dc, CPAU_dc, CPER_dc, DCs_dc, NPUL_dc, rXY_dc]=get_Gbim_data_DC()

keys = ['CDUR', 'CPAU', 'CPER', 'CDC', 'NPUL', 'rXY']
data_cpf = dict(zip(keys, [CDUR, CPAU, CPER, DCs, NPUL, rXY]))
data_dc = dict(zip(keys, [CDUR_dc, CPAU_dc, CPER_dc, DCs_dc, NPUL_dc, rXY_dc]))

data_all = {key: np.concatenate((data_cpf[key], data_dc[key])) for key in keys}

## Interpolate the field

In [ ]:
max_len = 1000
step = 5

duri0 = paui0 = np.arange(0, max_len + 0, step)
duri, paui = np.meshgrid(duri0,paui0)

CDURinterp=np.concatenate((CDUR,[0, 0]))
CPAUinterp=np.concatenate((CPAU,[0, 1000]))
rXYinterp=np.concatenate((rXY,[0, 0]))

points = np.array([CDURinterp + np.random.randn(*CDURinterp.shape)/1_000, CPAUinterp+ np.random.randn(*CPAUinterp.shape)/1_000]).T
new_points = np.array([duri, paui]).T.reshape((-1, 2))

cpf = interpolate_to_points(points, np.clip(rXYinterp, 0, np.inf), new_points, interp_type='natural_neighbor', minimum_neighbors=4, rbf_smooth=0)
cpf = np.maximum(cpf, 0)
cpf = cpf.reshape((len(duri), len(paui))).T

## Normalization

In [ ]:
max_b = np.max(data_all['rXY'])

## Plot behavioral tuning

In [ ]:
plt.figure(figsize=(13, 4.5))
plt.subplot(121)
plt.pcolormesh(duri0 / 1000, paui0 / 1000, cpf / np.nanmax(cpf), cmap='Reds', rasterized=True)
plt.axis('equal')
plt.axis('square')
plt.xlim(0, max_len / 1000)
plt.ylim(0, max_len / 1000)
plt.xlabel('Chirp duration [s]')
plt.ylabel('Chirp pause [s]')
plt.clim(0, 1)

plt.plot([0.03, 0.03], [0, 0.6], c='mediumorchid', label='min. chirp duration')
plt.plot([0.03, 0.6*0.8], [0.6, 0.6*0.2], c='royalblue', label='max. chirp period')
plt.plot([0, 0.6*0.8], [0, 0.6*0.2], c='seagreen', label='max. chirp duty cycle')
plt.legend(handlelength=0, handletextpad=0,
           markerscale=0, labelcolor='linecolor')

cb = plt.colorbar(shrink=0.2, label='Phonotaxis')

plt.subplot(2, 6, 4)
x, y, ys = p.transect_est(data_all, 'CPAU', [160, 240], 'CDUR')
y /= max_b
ys /= max_b
plt.plot(x/1000, y, 'ok', alpha=0.5)
plt.plot(x/1000, ys, '-k')
plt.xlabel('Chirp duration [s]')
plt.xlim(0, 1.000)
plt.xticks([0, 1])
plt.ylim(0, 1.09)
plt.yticks([0, 1])
plt.ylabel('Phonotaxis')
plt.axvline(0.04, c='mediumorchid')

plt.subplot(2, 6, 5)
x, y, ys = p.transect_est(data_all, 'CPAU', [160, 240], 'NPUL')
y /= max_b
ys /= max_b
plt.plot(x, y, 'ok', alpha=0.5)
plt.plot(x, ys, '-k')
plt.xlabel('Number of pulses')
plt.xticks([1,2,4,8, 12, 16, 20])
plt.xlim(0, 21)
plt.ylim(0, 1.09)
plt.yticks([0, 1], labels=[])
plt.axvline(1.5, c='mediumorchid')

plt.subplot(2, 6, 6)
x, y, ys = p.transect_est(data_all, 'CDUR', [120, 200], 'CPAU')
y /= max_b
ys /= max_b
plt.plot(x/1000, y, 'ok', alpha=0.5)
plt.plot(x/1000, ys, '-k')
plt.xlabel('Chirp pause [s]')
plt.xlim(0, 1)
plt.xticks([0, 1])
plt.ylim(0, 1.09)
plt.yticks([0, 1], labels=[])

plt.subplot(2, 6, 10)
x, y, ys = p.transect_est(data_all, 'CDC', [0.4, 0.6], 'CPER')
y /= max_b
ys /= max_b
plt.plot(x/1000, y, 'ok', alpha=0.5)
plt.plot(x/1000, ys, '-k')
plt.xlabel('Chirp period [s]')
plt.xlim(0, 1.3)
plt.xticks([0, 1])
plt.ylim(0, 1.09)
plt.yticks([0, 1])
plt.ylabel('Phonotaxis')
plt.axvline(0.6, c='seagreen')

plt.subplot(2, 6, 11)
idx = np.where(np.logical_and(data_all['CPER']>240, data_all['CPER']<320))[0]
x, y, ys = p.transect_est(data_all, 'CPER', [240, 320], 'CDC')
y /= max_b
ys /= max_b
plt.plot(x, y, 'ok', alpha=0.5)
plt.plot(x, ys, '-k')
plt.xlabel('Chirp duty cycle')
plt.xlim(0, 1)
plt.xticks([0, 1])
plt.ylim(0, 1.09)
plt.yticks([0, 1], labels=[])
plt.axvline(0.15, c='mediumorchid')
plt.axvline(0.8, c='royalblue')

p.label_axes()
plt.tight_layout()
plt.savefig(f'{FIGURES}/fig_2_behavior.pdf', bbox_inches='tight')

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import plotting as p

from metpy.interpolate import interpolate_to_points
import jax.numpy as jnp

from Gbim_import_dat import get_Gbim_data_chirpfield, get_Gbim_data_DC
from stim_gen import makeCHIRPstim

## Compare model and behavior on a common axis

In [ ]:
def r_upsample(x_dense, y_dense, x_sparse, y_sparse):
    """
    Remove NaNs from the dense vector, restrict it to the range covered by
    the sparse vector, interpolate the sparse vector onto the dense x-values,
    and MSE with the dense data.
    """
    x_dense = np.asarray(x_dense)
    y_dense = np.asarray(y_dense)
    x_sparse = np.asarray(x_sparse)
    y_sparse = np.asarray(y_sparse)

    valid_dense = ~(np.isnan(x_dense) | np.isnan(y_dense))
    x_dense = x_dense[valid_dense]
    y_dense = y_dense[valid_dense]

    valid_range = (x_dense >= x_sparse.min()) & (x_dense <= x_sparse.max())
    x_dense = x_dense[valid_range]
    y_dense = y_dense[valid_range]

    y_sparse_interp = np.interp(x_dense, x_sparse, y_sparse)

    r = np.mean((y_dense - y_sparse_interp) ** 2)
    return r, x_dense, y_dense, y_sparse_interp

In [ ]:
MAX = 20
step = 5
cpauMax = 1000 + MAX
max_len = cpauMax

## Load behavior

In [ ]:
[CDUR, CPAU, CPER, DCs, NPUL, rXY] = get_Gbim_data_chirpfield()
[CDUR_dc, CPAU_dc, CPER_dc, DCs_dc, NPUL_dc, rXY_dc] = get_Gbim_data_DC()

keys = ["CDUR", "CPAU", "CPER", "CDC", "NPUL", "rXY"]
data_cpf = dict(zip(keys, [CDUR, CPAU, CPER, DCs, NPUL, rXY]))
data_dc = dict(zip(keys, [CDUR_dc, CPAU_dc, CPER_dc, DCs_dc, NPUL_dc, rXY_dc]))

data_all = {key: np.concatenate((data_cpf[key], data_dc[key])) for key in keys}

## Interpolate behavior

In [ ]:
duri0 = paui0 = np.arange(0, max_len, step)
duri, paui = np.meshgrid(duri0, paui0)

CDURinterp = np.concatenate((CDUR, [0, 0]))
CPAUinterp = np.concatenate((CPAU, [0, 1000]))
rXYinterp = np.concatenate((rXY, [0, 0]))

points = np.array(
    [
        CDURinterp + np.random.randn(*CDURinterp.shape) / 1_000,
        CPAUinterp + np.random.randn(*CPAUinterp.shape) / 1_000,
    ]
).T
new_points = np.array([duri, paui]).T.reshape((-1, 2))

cpf = interpolate_to_points(
    points,
    np.clip(rXYinterp, 0, np.inf),
    new_points,
    interp_type="natural_neighbor",
    minimum_neighbors=4,
    rbf_smooth=0,
)
cpf = np.maximum(cpf, 0)
cpf = cpf.reshape((len(duri), len(paui))).T

## Generate the chirp stimulus grid

In [ ]:
input_stim, s, cpau, cdur = makeCHIRPstim(
    cpauMax=1000 + MAX,
    cdurMax=1000 + MAX,
    cpauMin=MAX,
    cdurMin=MAX,
    ppau=20,
    pdur=20,
    Fs=1000,
    num_steps=32,
    stim_len=5_000,  # Historical Fig. 2 duration; the Methods contain conflicting durations.
)
input_stim = jnp.array(input_stim)
points = np.array(
    [
        s["cpau"] + np.random.randn(*s["cpau"].shape) / 100,
        s["cdur"] + np.random.randn(*s["cdur"].shape) / 100,
    ]
).T

xi0 = yi0 = np.arange(0, cpauMax, step)
xi, yi = np.meshgrid(xi0, yi0)
new_points = np.array([xi, yi]).T.reshape((-1, 2))

cduri = xi.copy()
cpaui = yi.copy()

## Run the neural model

In [ ]:
from models.fix import Params, run_model

model_output = run_model(input_stim, Params(), verbose_output=False)
resp_ln4 = model_output[-1]
resp_ln4 = np.nanmean(resp_ln4, axis=0)
s["rXY"] = resp_ln4

cpf_model = interpolate_to_points(
    points, resp_ln4, new_points, interp_type="rbf", rbf_func="linear", rbf_smooth=20
)  # , minimum_neighbors=1)
cpf_model = cpf_model.reshape((len(xi), len(yi)))

## Normalization

In [ ]:
max_b = np.max(data_all["rXY"])
max_m = np.max(s["rXY"])

## Plot model–behavior comparisons and MSE

In [ ]:
r = []
fig = plt.figure(figsize=(13, 4.5))

plt.subplot(121)
plt.pcolormesh(
    duri0 / 1000,
    paui0 / 1000,
    cpf_model / np.max(cpf_model),
    cmap="Greys",
    rasterized=True,
)
plt.axis("equal")
plt.axis("square")
plt.xlim(0, max_len / 1000)
plt.ylim(0, max_len / 1000)
plt.xlabel("Chirp duration [s]")
plt.ylabel("Chirp pause [s]")
plt.clim(0, 1)
plt.plot([0.03, 0.6 * 0.8], [0.6, 0.6 * 0.2], c="royalblue", label="max period")
plt.plot([0.03, 0.03], [0, 0.6], c="mediumorchid", label="min. chirp duration")
plt.plot([0, 0.6 * 0.8], [0, 0.6 * 0.2], c="seagreen", label="max DC")
plt.legend(handlelength=0, handletextpad=0, markerscale=0, labelcolor="linecolor")

cb = plt.colorbar(shrink=0.2, label="Phonotaxis")

plt.subplot(2, 6, 4)
xb, yb, ybs = p.transect_est(data_all, "CPAU", [160, 240], "CDUR")
plt.plot(xb / 1000, p.nmax(ybs, value=max_b), "-k", label="Behavior")

x, y, ys = p.transect_est(s, "cpau", [160, 240], "cdur")
plt.plot(x / 1000, p.nmax(ys, value=max_m), "-r", label="Model (full)")

r.append(r_upsample(x, p.nmax(ys, value=max_m), xb, p.nmax(ybs, value=max_b))[0])

plt.xlabel("Chirp duration [s]")
plt.xlim(0, 1)
plt.ylim(0, 1.09)
plt.xticks([0, 1])
plt.yticks([0, 1])
plt.legend(handlelength=0, handletextpad=0, markerscale=0, labelcolor="linecolor")
plt.ylabel("Phonotaxis")
plt.axvline(0.04, c="mediumorchid")

plt.subplot(2, 6, 5)
xb, yb, ybs = p.transect_est(data_all, "CPAU", [160, 240], "NPUL")
plt.plot(xb, p.nmax(ybs, value=max_b), "-k", label="Behavior")

x, y, ys = p.transect_est(s, "cpau", [160, 240], "npul")
plt.plot(x, p.nmax(ys, value=max_m), "-r", label="Model (full)")

r.append(r_upsample(x, p.nmax(ys, value=max_m), xb, p.nmax(ybs, value=max_b))[0])

plt.xlabel("Number of pulses")
plt.xticks([1, 2, 4, 8, 12, 16, 20])
plt.xlim(0, 21)
plt.ylim(0, 1.09)
plt.yticks([0, 1], labels=[])
plt.axvline(1.5, c="mediumorchid")

plt.subplot(2, 6, 6)
xb, yb, ybs = p.transect_est(data_all, "CDUR", [120, 200], "CPAU")
plt.plot(xb / 1000, p.nmax(ybs, value=max_b), "-k", label="Behavior")

x, y, ys = p.transect_est(s, "cdur", [120, 200], "cpau")
plt.plot(x / 1000, p.nmax(ys, value=max_m), "-r", label="Model (full)")

r.append(r_upsample(x, p.nmax(ys, value=max_m), xb, p.nmax(ybs, value=max_b))[0])

plt.xlabel("Chirp pause [s]")
plt.xlim(0, 1)
plt.ylim(0, 1.09)
plt.xticks([0, 1])
plt.yticks([0, 1], labels=[])

plt.subplot(2, 6, 10)
xb, yb, ybs = p.transect_est(data_all, "CDC", [0.4, 0.6], "CPER")
plt.plot(xb / 1000, p.nmax(ybs, value=max_b), "-k", label="Behavior")

x, y, ys = p.transect_est(s, "CDC", [0.4, 0.6], "CPER")
plt.plot(x / 1000, p.nmax(ys, value=max_m), "-r", label="Model (full)")

r.append(r_upsample(x, p.nmax(ys, value=max_m), xb, p.nmax(ybs, value=max_b))[0])

plt.xlabel("Chirp period [s]")
plt.ylabel("Phonotaxis")
plt.xlim(0, 1.3)
plt.ylim(0, 1.09)
plt.xticks([0, 1])
plt.yticks([0, 1])
plt.axvline(0.6, c="seagreen")

plt.subplot(2, 6, 11)
xb, yb, ybs = p.transect_est(data_all, "CPER", [240, 320], "CDC")
plt.plot(xb, p.nmax(ybs, value=max_b), "-k", label="Behavior")

x, y, ys = p.transect_est(s, "CPER", [240, 320], "CDC")
plt.plot(x, p.nmax(ys, value=max_m), "-r", label="Model (full)")

r.append(r_upsample(x, p.nmax(ys, value=max_m), xb, p.nmax(ybs, value=max_b))[0])

plt.xlabel("Chirp duty cycle")
plt.xlim(0, 1)
plt.ylim(0, 1.09)
plt.xticks([0, 1])
plt.yticks([0, 1], labels=[])
plt.axvline(0.15, c="mediumorchid")
plt.axvline(0.8, c="royalblue")


p.label_axes()
plt.tight_layout()
plt.savefig(f"{FIGURES}/fig_2_model.pdf", bbox_inches="tight")

In [ ]:
plt.figure(figsize=(3, 2.5))
plt.bar(np.arange(5), np.array(r), color="lightgray")
plt.xticks(np.arange(5), ["Duration", "Number", "Pause", "Period", "DC"], rotation=90)
plt.ylabel("MSE")
plt.ylim(0, 0.1)
plt.yticks([0, 0.1])
plt.tight_layout()
plt.savefig(FIGURES / "fig_2_mse.pdf", bbox_inches="tight")